# 13 . Search Lab — sweep strategy variants without fooling yourself

Drives [`experiments/search_harness.py`](../experiments/search_harness.py). Findings live in
[`docs/STRATEGY_SEARCH_LOG.md`](../docs/STRATEGY_SEARCH_LOG.md).

**The harness enforces five rules, each of which came from a result that looked good and wasn't:**

| Rule | The result it came from |
|---|---|
| costs always on | a rule at +0.025R died to 0.7 bp of spread |
| score 3 regimes, judge the **worst** | short-only: +294% in a −46% year, −43% in a +26% one |
| `allow_leverage` **and** `max_notional_pct` | risk sizing silently capped at ~1×; realised risk 0.25%, not 1% |
| **outlier guard** (`expR_ex_top`, `med_R`) | one +202R trade carried a whole window |
| hold-out selection | a filter ranked top-decile in-sample, **110/127** out-of-sample |

> Read `expR_ex_top_min` and `med_R`, never `expR` alone.

In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd
from quant import scenarios as sc
from quant.data.timeframe import resample_ohlcv
from experiments.search_harness import (make_windows, search, score, expand,
                                        select_and_verify, r_stats, CFG_LEVERED, SPREAD_BTC)
pd.set_option('display.width', 260); pd.set_option('display.max_rows', 120)
print('scenarios  :', sorted(sc.SCENARIOS))
print('stops      :', sorted(sc.STOP_METHODS))
print('tps        :', sorted(sc.TP_METHODS))

## 1 . Regime windows
One bull and two bears, so nothing can score well just by being pointed the right way.

In [ ]:
W = make_windows()          # 1m; pass spans={...} for other periods
TF = '15min'                # execution timeframe: '1min' | '5min' | '15min'
if TF != '1min':
    W = {k: resample_ohlcv(d, TF).reset_index(drop=True) for k, d in W.items()}
for k, d in W.items():
    print(f'{k}: {len(d):>7,} {TF} bars   BTC {d.close.iloc[-1]/d.close.iloc[0]-1:+.1%}')

## 2 . Sweep a grid

In [ ]:
PARAMS = dict(grad_period=200, grad_smooth=9, grad_lookback=5,
              grad_max=0.0, grad_min=0.0,          # short needs falling, long needs rising
              sr_left=10, sr_right=10, sr_cluster_pct=0.15, sr_min_touches=1,
              prev_low_n=20, prev_high_n=20, confirm_red=2, confirm_green=2,
              stop_buffer_pct=0.05, stop_pct=0.6, stop_n=20, stop_atr_mult=1.5,
              tp_rr=2.0, trail_pct=1.0, tp_opp_n=1, sec_opp_n=1)

grid = []
for scen in ['support_break_short@v1.0', 'support_break_short@v1.2',
             'resistance_break_long@v1.0', 'resistance_break_long@v1.2']:
    for stop in ['leg_high@v1', 'high_of_n@v1', 'atr@v1']:
        for tp in ['rr@v1', 'trailing@v1', 'opposite_candle@v1']:
            grid.append(dict(scenario=scen, stop=stop, tp=tp, params=PARAMS,
                             tag=f"{scen.split('@')[0][:4]}{scen[-4:]}|"
                                 f"{stop.split('@')[0]}|{tp.split('@')[0]}"))
print(f'{len(grid)} configs x {len(W)} regimes')
res = search(W, grid)
COLS = ['tag', 'n_total', 'expR_min', 'expR_ex_top_min', 'medR_min',
        'n_windows_pos', 'n_windows_pos_ex_top', 'top_R_max', 'ret_min', 'dd_max']
res[COLS].head(12)

## 3 . Survivors only
Positive in **every** regime *after removing each window's single best trade*. Anything that fails
this was carried by an outlier.

In [ ]:
good = res[(res.n_windows_pos_ex_top == len(W)) & (res.expR_ex_top_min > 0)]
print(f'{len(good)} of {len(res)} survive '
      f'(by chance you would expect roughly {len(res) * 0.5**len(W):.0f})')
good[COLS] if len(good) else 'NONE'

## 4 . Parameter sensitivity — plateau or spike?
**The single most useful check here.** A real edge degrades smoothly as you detune it. If expectancy
flips sign between adjacent settings, the config is sitting on noise.

In [ ]:
CAND = dict(scenario='support_break_short@v1.0', stop='high_of_n@v1', tp='rr@v1')

for name, axis in [('tp_rr', [1.0, 1.5, 2.0, 2.5, 3.0]),
                   ('stop_n', [10, 15, 20, 30, 40]),
                   ('grad_max', [None, 0.0, -0.02, -0.05]),
                   ('sr_min_touches', [1, 2, 3])]:
    cells = []
    for v in axis:
        r = score(W, params={**PARAMS, name: v}, **CAND)
        cells.append(f"{str(v):>6}: {r['expR_min']:+.4f}")
    print(f'{name:16s} ' + '   '.join(cells))
print('\n(reading: a smooth run of similar numbers = plateau = plausible.'
      '  a sign flip between neighbours = spike = overfit.)')

## 5 . Hold-out selection
Rank on TRAIN, then report the untouched TEST result **and where that pick lands among all
candidates on TEST**. A trained pick that is not near the top of the test ranking is noise.

In [ ]:
from quant.data import get_ohlcv
d = get_ohlcv('BTCUSDT', '1m', start='2025-01-01', end='2026-07-29', source='binance', tz='UTC')
if TF != '1min':
    d = resample_ohlcv(d, TF).reset_index(drop=True)
TRAIN = {'train': d[d.t < '2026-01-01'].reset_index(drop=True)}
TEST  = {'test':  d[d.t >= '2026-01-01'].reset_index(drop=True)}
verdict = select_and_verify(TRAIN, TEST, grid, top_k=5)
verdict

## 6 . Risk control — daily / session loss cap
`daily_loss_limit` = % of session-start equity; once a session's **realised** loss passes it, no new
trade opens that session (open trades keep running). It cannot prevent the trade that breaches it.

Measured: it lowers drawdown consistently but does **not** add expectancy — a risk control, not alpha.

In [ ]:
rows = []
for cap in [0.0, 1.0, 2.0, 4.0]:
    for wname, w in W.items():
        r = sc.run_pieces(w, CAND['scenario'], CAND['stop'], CAND['tp'], params=PARAMS,
                          cfg={**CFG_LEVERED, 'spread': SPREAD_BTC, 'daily_loss_limit': cap})
        st = r_stats(r)
        rows.append(dict(cap=('off' if not cap else f'{cap:.0f}%'), window=wname,
                         n=st['n'], expR=st['expR'], ret=st['ret'], dd=st['dd']))
pd.DataFrame(rows).pivot(index='window', columns='cap', values=['expR', 'dd', 'n'])

## 7 . How to judge anything you find

Before believing a config, all of these must hold:

1. positive in **every** regime, **after** removing each window's best trade;
2. `med_R` not pinned at −1.0 (otherwise it is a lottery ticket, not a tradeable edge);
3. a **plateau** in §4, not a spike;
4. the trained pick lands near the top of the **test** ranking in §5;
5. enough trades — a few hundred minimum — that ±0.05 expR is not noise;
6. drawdown you could actually survive at the leverage you intend.

Nothing found so far clears all six. The closest is `15m | v1.0 | high_of_n | rr(2R)`, which passes
1, 3(partly), 4 and 6 but fails on sample size and shows parameter spikes — see §5 of the search log.